In [5]:
import sys
sys.path.append("..")

In [32]:
from securerag.models.fid import FiDT5

model_cls = FiDT5
model_path = "../models/nq_reader_base"
model: FiDT5 = model_cls.from_pretrained(model_path)


In [23]:
from securerag.models.rag_seq import RAGSequence


model_cls: RAGSequence = RAGSequence
checkpoint_path = "../models/rag-sequence-nq"
# retriever = transformers.RagRetriever.from_pretrained(
#     checkpoint_path, n_docs=self.config.n_context
# )
model: RAGSequence = model_cls.from_pretrained(checkpoint_path, n_docs=10)

/root/miniconda3/envs/securerag/lib/python3.8/site-packages/transformers/modeling_utils.py:927: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(resolve

In [33]:
print(model.state_dict)

<bound method Module.state_dict of FiDT5(
  (shared): Embedding(32128, 768)
  (encoder): EncoderWrapper(
    (encoder): T5Stack(
      (embed_tokens): Embedding(32128, 768)
      (block): ModuleList(
        (0): CheckpointWrapper(
          (module): T5Block(
            (layer): ModuleList(
              (0): T5LayerSelfAttention(
                (SelfAttention): T5Attention(
                  (q): Linear(in_features=768, out_features=768, bias=False)
                  (k): Linear(in_features=768, out_features=768, bias=False)
                  (v): Linear(in_features=768, out_features=768, bias=False)
                  (o): Linear(in_features=768, out_features=768, bias=False)
                  (relative_attention_bias): Embedding(32, 12)
                )
                (layer_norm): T5LayerNorm()
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (1): T5LayerFF(
                (DenseReluDense): T5DenseReluDense(
                  (wi): Linear(

In [25]:
import transformers
import torch
from securerag import data

path = "../data/open_domain_data/NQ/dev_with_scores.json"
datas = data.load(path=path, size=1000)
dataset = data.Dataset(data=datas, n_context=10)

tokenizer: transformers.T5Tokenizer = transformers.T5Tokenizer.from_pretrained(
    "../models/t5-base", return_dict=False
)
data_loader = torch.utils.data.dataloader.DataLoader(
    dataset=dataset,
    batch_size=10,
    collate_fn=data.FiDT5Collator(
        tokenizer=tokenizer,
        text_maxlength=50,
        answer_maxlength=50,
    ),
)
record1 = next(iter(data_loader))

/root/miniconda3/envs/securerag/lib/python3.8/site-packages/transformers/tokenization_utils_base.py:1767: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(
/root/miniconda3/envs/securerag/lib/python3.8/site-packages/transformers/tokenization_t5.py:184: UserWarning: This sequence already has </s>. In future versions this behavior may lead to duplicated eos tokens being added.
  warnings.warn(


FiDT5Collator elapsed: 0.2484 seconds


In [26]:
(
    context_ids,
    context_masks
) = (
    record1.passage_ids,
    record1.passage_masks
)

In [34]:
import torch

total_params = sum(p.numel() for p in model.parameters())
print(f"总参数量: {total_params/1e6:.2f} M")


总参数量: 222.90 M


In [35]:
state_dict = model.state_dict()

total_params = sum(p.numel() for p in state_dict.values())
encoder_params = 0
decoder_params = 0

for name, param in state_dict.items():
    # 跳过 embedding 部分 (包含 embed 或 shared)
    if "embed" in name or "shared" in name:
        continue

    if "encoder" in name:
        encoder_params += param.numel()
    elif "decoder" in name:
        decoder_params += param.numel()

print(f"Encoder 参数: {encoder_params/1e6:.2f} M ({encoder_params/total_params:.2%})")
print(f"Decoder 参数: {decoder_params/1e6:.2f} M ({decoder_params/total_params:.2%})")


Encoder 参数: 84.95 M (28.61%)
Decoder 参数: 113.28 M (38.15%)


In [36]:
import torch.nn as nn
from thop import profile

class GenerateWrapper(nn.Module):
    def __init__(self, model, max_len):
        super().__init__()
        self.model = model
        self.max_len = max_len

    def forward(self, x, masks):
        return self.model.generate(x, masks, 50,)

wrapper = GenerateWrapper(model, max_len=20)

macs, params = profile(wrapper, inputs=(context_ids, context_masks))
print(macs, params)


[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.


FiDT5.Encoder elapsed: 2.8011 seconds
FiDT5.forward elapsed: 0.5138 seconds
FiDT5.forward elapsed: 0.1195 seconds
FiDT5.forward elapsed: 0.1211 seconds
FiDT5.forward elapsed: 0.1189 seconds
FiDT5.forward elapsed: 0.1208 seconds
FiDT5.forward elapsed: 0.1199 seconds
FiDT5.forward elapsed: 0.1195 seconds
FiDT5.forward elapsed: 0.1251 seconds
FiDT5.generate elapsed: 4.2119 seconds
505353338880.0 222855168.0


In [15]:
import torch
import time

N = 768

# 生成 CPU 矩阵
A_cpu = torch.rand(640, 200, 768, dtype=torch.float32)
B_cpu = torch.rand(N, N, dtype=torch.float32)

# CPU 计算时间
start = time.time()
C_cpu = torch.matmul(A_cpu, B_cpu)
cpu_time = time.time() - start
print(f"CPU 时间: {cpu_time:.6f} 秒")

# GPU 全流程拆解计时
# 1. CPU -> GPU
B_gpu = B_cpu.to('cuda')
start = time.time()
A_gpu = A_cpu.to('cuda')
torch.cuda.synchronize()
cpu_to_gpu_time = time.time() - start

# 2. GPU 计算
start = time.time()
C_gpu = torch.matmul(A_gpu, B_gpu)
torch.cuda.synchronize()
gpu_compute_time = time.time() - start

# 3. GPU -> CPU
start = time.time()
C_result = C_gpu.to('cpu')
torch.cuda.synchronize()
gpu_to_cpu_time = time.time() - start

# 总流程时间
gpu_total_time = cpu_to_gpu_time + gpu_compute_time + gpu_to_cpu_time

print(f"GPU CPU->GPU 时间: {cpu_to_gpu_time:.6f} 秒")
print(f"GPU 计算时间: {gpu_compute_time:.6f} 秒")
print(f"GPU GPU->CPU 时间: {gpu_to_cpu_time:.6f} 秒")
print(f"GPU 全流程时间: {gpu_total_time:.6f} 秒")


CPU 时间: 0.357085 秒
GPU CPU->GPU 时间: 0.100634 秒
GPU 计算时间: 0.012799 秒
GPU GPU->CPU 时间: 0.379251 秒
GPU 全流程时间: 0.492684 秒
